# GAVE2 CMRRWNet V3: L4 Production Run

This notebook trains a clean, native-resolution five-fold CMRRWNet v3 run, selects probability calibration and TTA from honest OOF predictions, and creates one certified preliminary submission. Run the cells in order on a Colab L4 runtime. The Drive run is resumable only when its v3 manifest and fold configurations match exactly.

In [ ]:
from pathlib import Path
import base64
import hashlib
import json
import math
import shutil
import subprocess
import sys
import time
import zipfile
import zlib

TEAM_ID = "梯度不下降队"
RUN_NAME = "gave2_cmrrwnet_v3"
EXPECTED_ARCHIVE_SHA256 = "7ca8066fd93d590dc606134046dd662edd522d8b4c0c0aa40545393371cc1cc1"
COMPATIBLE_ARCHIVE_SHA256 = set(["7ca8066fd93d590dc606134046dd662edd522d8b4c0c0aa40545393371cc1cc1", "d72047b09cdb4ce3d0d74c1c5971a79430ce8354db9890f5ac5f10e4ea442b00"])
DRIVE_BASE = Path("/content/drive/MyDrive/MICCAI2026")
ARCHIVE_PATH = DRIVE_BASE / "miccai_cmrrwnet_v2.zip"
WORK_ROOT = Path("/content/MICCAI2026")
DATA_ROOT = WORK_ROOT / "GAVE2_preliminary"
RUN_DIR = DRIVE_BASE / "runs" / RUN_NAME
OOF_ROOT = DRIVE_BASE / "oof_cmrrwnet_v3"
OUTPUT_ROOT = DRIVE_BASE / "submissions_cmrrwnet_v3"
FINAL_ZIP = DRIVE_BASE / f"{TEAM_ID}_cmrrwnet_v3.zip"
FOLD_MANIFEST = RUN_DIR / "fold_manifest.json"
BASE_CHANNELS = {"task1": 24, "task2": 16}
FALLBACK_CHANNELS = {"task1": 16, "task2": 12}
NUM_REFINEMENTS = 2
MAX_EPOCHS = 120
SEED = 77

def run_module(module, *arguments):
    command = [sys.executable, "-m", module, *[str(value) for value in arguments]]
    print("RUN:", " ".join(command))
    return subprocess.run(command, cwd=WORK_ROOT, check=True)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
assert DRIVE_BASE.is_dir(), f"Drive folder is missing: {DRIVE_BASE}"
assert ARCHIVE_PATH.is_file(), f"Upload miccai_cmrrwnet_v2.zip to {DRIVE_BASE}"
print("Drive mounted and archive found.")

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()

actual_sha256 = sha256_file(ARCHIVE_PATH)
assert actual_sha256 in COMPATIBLE_ARCHIVE_SHA256, (actual_sha256, COMPATIBLE_ARCHIVE_SHA256)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    assert archive.testzip() is None
    names = archive.namelist()
    assert all(not Path(name).is_absolute() and ".." not in Path(name).parts for name in names)
    assert any(name.startswith("GAVE2_preliminary/") for name in names)
    assert any(name.startswith("experiments/gave2_ensemble/") for name in names)
if WORK_ROOT.resolve() != Path("/content/MICCAI2026"):
    raise RuntimeError(f"Refusing to replace unexpected path: {WORK_ROOT}")
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)
with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    archive.extractall(WORK_ROOT)
assert DATA_ROOT.is_dir()
print({"archive_sha256": actual_sha256, "members": len(names), "work_root": str(WORK_ROOT)})

In [ ]:
EMBEDDED_SOURCE_PATCHES = {"experiments/gave2_ensemble/cmrrwnet_v2.py": "eNqVWEtv4zYQvvtXcNVDZcBREye9GHWBNkjay2YXwT4ORiAwEm2zkUiVpOwEaf57Z0i9SFtJNkAsi5wZzsw3L3qtZEnSdF2bWrE0JbyspDKECiENNVwKPZk0a+5R8PukNryYrJGzomYLKy3bZ3jt6I1U2dZ7SYQgVBMhHLNbQ2E6ybYse6gkF+05pF+ZTCY5WxP4p3VhUrle84zTItWyVhmLp+Tkd3vyYkLgTzEwRZDYvuAfbsVgIi/AwGmimJbFDtiSiiomjF7N7zraX0j0IOS+YPmGpfdUs2i45Q7U3tqGm2197y19ZmIzPz29SC8/3t5+v2HG2zWKcuGtlDJn4ILqya1OG3sLSfPeWCCqCxY7FRbWKPIf0UbB540UjCztY+qc4MhShAc2rAfc0pTwdbNLOEAhjeMGBdi4h61MYETygeiEa+vWuDnVup9ykHQNqzfSXMta5FdKSRWvo0+NVNK6ZaCHYv/WXLF8QZ4H8l8id7KuWAZm+AGY4GqKgeSgLWRmAzaONnTH5mlWKrUXzKS7eWdQNBuq35ll5YMW1hNS2fcE3c9UuxwaeFsLw0vW2nYp6yK37skUo4ZBArVxjMI4HG+VI2sQf8xEB++hkW7dmYmSYvxwLEY99VoNVE7YIzimiRf3cAzsMWOVIVf2gcpALsLaG6ZdU/BuDsnaGiTHcPTsAiRBNphHbK7Dd3sM2i9oCaBDikZfgfuj1RCQiW7YfrAwiKkm8rZUU2NUY9PMyhlQjVrw6V0Ko3g4hjyj2BaWppi4E5u8zLa0KCDFWSoVuNthA2WKKUGLuP0C8QilQS+aKveFCS2VLVXDBad9FEEAiR0bOreVQ1ZUwbenGdkxLvBTa1bcIR6DHVxzFHcJSJu0kR1ok4icl+TDklxgmIebeksrtjq7Q4LzMCq+0aLuPHr1COFmICr+vPz7O3HsZA+VkJitYgxdJASUlBnZAG7Ppq4gFI8eNw08HRCtFjOyOp2R+Yyc3c0IvC3uAIasoFqTS6mUVeOWZbXSfMdagGMhEhdG087Dh1HgUG0Uz1phrfJEs5JCHGUakjkn91jKYFexNReshMaROC9/2YK9suKwhw2jgN2fsaA1KsFeziBXoSpogm4yW67yHt72NKpdUoO0NX8EYQ5U5IRQ328hCe3mX398uyJNbUK9dH1fcq0xnWvtSDbAIqy0VjhmHe4MZCagONS2vaJVBaIeGKu0pRlqXla1mwJcPAlrAKpuXdHZYPPAasMeK6nBpQyC+Ql0A9E7iJ6cVFDceWarDu90c2nk2JMWJ+dVTLQUsoqbNO1bOSi/nnVvhuqHBbbAfgkRSNvoW6CG/Z6oy7RHz+1CwZ33FBQU3Fl703764GKzAPRlAbRfVM168qBNjjdlx2KT3+8l1kEwiXSWTj3jgB0fUNf3SDash3YbiyKA8hzh2xmWUPwyj16OFUUvgb8KOBqrOVZ2FPWMnx9Um43NKZ47yW/k4g3Bkc9Q1tqQe2iGhhSMwvcLX3yACBxw+tYBIUt7hJDiRLANgLeD1jHppHR99eg0FSA4UE70VkAx7Dy+XJLG2W5i+rXjWHOlTZpZeic96ZvZKH9D6XU+L9oTJ7cGqd0J8UC7GTmf+TBNfX7nLHVMq/jcFtbXuAdR6G/4QC8xl+LXBIW4OY5gNeAZS0dgxnyMx/YDMQHIwA0JGtuZOMQfcQrJ3z8lTwelS9WiG9mxaDXeX5CuM0G7xrh+14CAfz+hCidY2uEGIYb3I3SJq9VQeeFSA/MLFGZoAzTn1tlQjQW0kKYpOGFY6nlJNziJASdWdtCT5JI5e5uJvJeS2E5ng5B8PcH+yXUgr6Bqw2C7Bwb0gkpTMoWDN5zR5WuvvZ0tZTKsCy708J6Eptku91o8BDXDDRI9RTcvWn/PsE2mnRuX1xQw7SPGG/hiyzGAFfronqq8gdS67wh+BddmNVy9WwRlouucS2dZGC5t1jdneEXTlzA2tI2P8xZAbRB5ZywA8Dwi9KWb5CBKtGE0h/RwM96wyLqxohnZukrVSsMZbr447y/Z/SyAxKvX52lf1vRugFQzp6QugJcNEJpvSsnzkLHXFoehFPsm4L9p/B0WIt+Rbs5OcbzujTwOXFNuZ6F6U08gwHgo04Ny7mswAucBjbUvuu0sOQD6lWNHwN7L6OCYwJo+mh0EcNeN48OTZn6gTGcEbiPLM1/YIDgSHE1FHr/vwjWdBmXgeHhY3fxIOdQUVPNJfMUPNW/KxkD5vmhANaY5NdQGiC0QSLPCuZXI+3/g0jGoD42gZ88W9wtRam89ERScp8pFGw6OeFtN05lPj/0aCLsWHmx7jbql8xYDhiA9WpZgOWAaK9ct99h+ICZosy13sNwzvbTXdPtDzPBnIJcuwa1h7Mbw+m3hB24KP3BLsMExfqv1ft985fLr3SGWPv6eucsRzAPbl6M4j7lh+Ta2gVuWR/GcTv4H6RVegQ==", "experiments/gave2_ensemble/losses.py": "eNrVWktv4zgSvvtXCDkMJEdW2w7SAxhIAz2NyV52ZoDsoPcQBARt0TYRmdJKlBM39sdv8U3q4Tyme4DtQ8emqopVxXrxk7d1eYgQ2ra8rQlCET1UZc0jzFjJMaclayYTvcbLerMPvmSMRbiJGOuuZtuWbQQ3LgTB7WSyFftkvMaUUbZDLadFg45Ls19NNm3d0CNBDcc7gp4I3e057D3JyTZCBW44qmqSUyk1dh+bZDWJ4B/dRrShDLjZhvjP0yguaMPTiLdVQRJNLv7VBGxmkUd7P1s8TIYfaU0OuHkkOVpvQEXK96god5Q3sfqz0vb/SVhT1rAjrneEd1eFiHAtiWafggWloyC8im7k34w8V5jlCDexkppIkqJsGqC4zdaU4fqENjUsIMJ4XVanAQ2NTilYmLfSspsLVjJyoeTlhMExqS2vsqY9xEm2KfChQgfK4kU2T3z3xHL7qaJOFHn0QQnR/mrKLUfgQ4IEbaCRNmDAcfLJoPOsX4bWSQWCtkWJOViwILOP6WTEs1VdroFGPWjo7lDSXHso0fZ4OghK9cF7lNOD8Hw8T6NlGl0pv1DGSd0Q6VfxUO4z1dzKQYIv9LUgco+iS03epQYfCpHLbA4Sg40uheXC8bESqb77BwUHF82kiOxAMIsTfTrbcoMLxI8g6/H0N5zQFhcNQVXZUC5SXSW5O7N5djX3CRnZ4RHCnzXhDh8OOHhw/Z5gwGtaUH76ETHB69aZbILC7DcWG6Gj+lyxOlHNrHUYFmKcKIRoNk9WMqqCjgrBFprgws1W0oDArl4OnzfsGK73GDrnbhnMuscAqshvQbirM6zKJ+MmZUqqoiUJc+BImoYUaAM1HtoEYRuVCCMl/dXF+1yJkQS4hhQ+6fy/X6XRfKWbj9LIe7JYLc0Tyrz15epKrZdgILICb7OaFG2sv8+0vMRRajGaTn4LqUx9j33Bl47ZRVzgSuR1x22B+Zva4iMh1as8C1pC4SM5WCB4ok+Q9dd2BmA8NgS6f+WE481eNLKqjZMkuoEy0RsClKa6hU1B4tx3xKv7673Z+8GY6VaEl6CXQsf8Y7ulG4qLX778evVPEWswMP1W5i2MJ0qxi4uLL7/d3f37d6gvawwSKCORoFYNf1vW0eev8AWvSdFkE8nzZQ8zG4RNWeekBj9EfE+bqMKbR5inYDaKPn/4+ueHr2qDOThPHStEF3zWIcdrQiCs5AJlmST9fFjTXVu2jSaaCaKoos+wNcgA0TtWwjQhtVIyP8iQatqK1EcYyqBFrVuuzg7GviJqG03uS/TIRZrxzR7mRDCCRKV2l3VFZpw00Y1UdLL6Cde5OPttGv2Qccw0toVN67psWR6LNZm/aQT/rx6STLadOLFcO46wbRg+qU9x5AHJYogkoFh2KcS0KjfSsTi0kaSRWzmixTBRQGM3s0Qw4JOaY1lJYqX/TCkp/2B/cJybwVH8e2TlE0NYyI+VP2dOWI/LsulRt1djlNWp3DO1wpMun/xzOcJ+lOzHd7Nzxc9VNC2SfnVpml72392N5f6duQwB/xbi/QAFR2kgM2d9CtOiVyqyIDUQgisXR0jmhozt32HgdzEtEw/Ko6Vz6guOTEiFwkchs9VY2ytfyVgeBtew75B+UODhWvrmi553EgMWxQNaKr3CKEAi8V/gv58/jIkA3QsS3lxFK1oMqqn2C4MfLV+x/+Kt+y/P7A8Brza2JKJoU1NfRZMhrD2QGvPgJO6XK1Bj2TkBa4P+cBlR6LRDBnXvqsoQV0CEDNKxG+quJYB+dMAF/QZ9UFwJrmGbRzE0P8Kew8np2QqDreO3SfuPz19/Xf6L7EQqSlhkMHeDdAuyKLXfJHTQuc/AmOoI5G35LIU/q/avRovr1DPR1A+Ukw0+Dd2U3lMSrIYgyX0JiTw7gMr7FpL1jQHq/mLI1LULWLpLXgGEcrEryI+eDXSn6Dpoeh4zOldxXN/pOXR6Dll5s+SBM5i+cDHqSg063f9VNzDh8co2QITz7t2wpJHKcPF1dbKjlJaU4aoiTDmtH+hTqGO9Oj6DUjqD0pb0Ci4JxRlbByPEcfOS48Jlo7gVaeV6B+6eQcjY24laSdUCmP+NVkZCqvVKBITgb6QvkQ6coKRBAjY2Ye2ZHMZI9N9InPW9v/YwEjgBRGDu5WfxZXO7VVpCIo/k83dHyzq4SU/qiIUuILsCMl7GOTlCzbjR9131LY1yfqrcoviSZEdKnmK4G8wW8oqgG+crQGkPtpoZZaYOLrvNYJ8hRMRDqmY9gCvgmvlsFqzwECojKumj02egbXXO5HlTtPJVBD4i2f0pzLuy6Z9Dhv5iQbPojXdLXC0ePHRDozbeBXB15T+3etv7qVBUALzPZR2b276Cb35SVy/FnUy8arouyyK2ojLMTvHAO5MRuAQcpkueUQH8FsfefXQlBlTvWgkmJKkATm90fClMwyAQRcl2cE7Nf1pCvpFY03gIkNO0Q2ORmwCyia2GGUw4hxbqsQZrRYD76I3SI0RvPLhwT6Gag4HfDyy0Ne/1qKFlGYcPA5IhHDEgsIDikZaFevs3jidCmr2MIHqCfJDaJOK5PERywoFSWxAuS7IFrbt+FvcGrPsCZaJjXY042r6SInWZk/iIi5Z40b0va/qthAlfuG92mx3wM6ogIZZQciQtTBUyUtKo4TXNyQ0UxgrnOXTVG1Er514TPZKai/QbF7UQLwkGRc3ToJsHTQvcRA/gNqdsavfybuLSzBJaf89KabtInp4nuhsGais2mSp9jRd6Z3NcLmz8FxczTyv/JYTaWOoQhOVpYoYpJKaIGrMdid1pezYZ3nGTclJw7KHekmHWc5N303Gm2I825JW0mXsyVRsE0W8e+u9CN4Wd2f9iI3lL3L/8jsvUh+Bd104inIjXLd+H6CRQ+oQSBvMcNp67vtY33lF6M9IZQb5CZyWBQhsFPct3Z75208AsV48Cqn5hIrOPpvswOWcoX8ZdnafB6zUrvUN2Rr6KEPeu15ky9feWb+Dcs8vg2aDg4GWw2sXiG7/gQtygcgs9vhvf6A6d55GPeba8fgH6CBEN87K6T/XRR0jCse28SNfL+3Rzf+uyKiFZBsneDaXUZCdu1zVat9stUF50/XeR6nzFDeIypeOei/X8ruikVlfL5Bxmo95QuKXkHHSjiL21DnV4JJYhXO7wDB6QZR182pHQPTXL3H3Q1TY8RaduuJ74+JH4SdKPho/WMudfumQOXdxTZVg3KpIgrUxvfA9i5H4K8MIPRs6LCSB4H8oSpg9iXFLxy8Eomxqt/i60SQaBBM4gX0P45dW/PYvgSkGie49iCEca+S2chH2UFkkPzQtgGA31yKgdPJIORmNAqgCl6e20pUwOs+qB/ancGKQ4mODTl2/Xcps3YJe9OjAduaNpwb23IoNV4ZP/G4GRnbs80/6Ed9YYHzL9H58pWMI=", "experiments/gave2_ensemble/train_v2.py": "eNrNPGuP28iR3+dX8BgcQtoSrZlZe3PKaYHN3eYQYLNZOIvNB2HQ6KFaI2b4OpIaW2v4v6eq+t0kNRonuYuxWEvdVdVV1dX16pb3XVNFjO2Pw7ETjEVF1TbdEPG6bgY+FE3dX13pse6h5V0v9PeHXH/6a9/U+nPFh4P+3PF611RXe1wjb+pBfBzK4l6vUR/LUo1KkBZQnfkfkdKVAa7aU8T7qG6vJHS24wPXsP/z7c/f3fw3DPRiWMBaVXscBOuaguUHkEWUrAdx+kVUFv3AcgBjxQ6+Dk2XH1jelCUfhKJbAEsPXTGc2NONpp9cRfBn35Q7VvG62Aug0h/4zdt3C5o58P7A9kUpnK8oDqvEwJFROV42fMc8KnL8iZcFQImpuQ/ACohyrJ3xVLE6dLyoi/qBHYei7B1++ZPo+INgbSfyoodtZIjADiB989DxCkRvm74YiifBPoji4TD0EiJvjvUAe361E/uISWqMtJSka+Jn6E7yA/5RyxGAGSTeaCgjvrydwk36HtQguiuD0AmwvlqiLFwInBQfc9EO0R8I/buuazo0AxhdR9GvohbErPg6qhvYdRA6Wkaifiq6pq5EPSz7FuTfF7ldihe9iN6DkEUliFoS/9cf37//yw9iiEB/WqPA0/8ei0700Y+nn5CvOJVywcJKO2BqrBdi1yf4/3UEdpNGy2+iH5paSA1J+89wmmBSGq3bbHpCyc+iTah5O52BERx5yQLUYq+m8+OOZ0XP+BMvSn5fCr1tlgKBOGQYL0tFSgrGj0MDRwTOiTydiWJsJ56KXKyjfugWEa9a+qTIAwNyOoOD1g39h2I4JDEuBYoDaRE+2myi+H5//S5eT259phdOJCk2nFqxkURgdfomAe/3cJKG63dSeEXD8SeJFeUB7QD8W1nCsa+feJ/c8yE/rKNdkYOrAI/xKOXArcOx9SV7UVRwunqYJGLbWH6P72iyAprOHH1VU6CbBzE4k2pATYMWlWnIv5I0+s9olb11D5xaWCpiXxZtIsdAQ0XVb5Ll9SJNDbzmxQGnoWloy54DrwbHGP8Qbm9exO3Ni7klDGkmD/cAKpffrhfR+vas0n9jxXgAnwCoCuoIXrjpqmSV/cdqEV1n1yvL0n3B+zHkcpWtbhcR/j91iFYVv4yq5DzBv15JZl7TSmmWl3CuAIdQVmnWNh8SInx2h26cLXJov1Z6RNialcWjwOEU1hzxuCJprtMRB1cT257zIUFKC0f5t+u7lPZoc50im3Img5jaiu31XfRNdBuJElw14BkPB6ZAPgQ/3MTkVqbEu15Z8VT4B15uZ+1VLvRVYLXIpcZem0+vI2Bug8JLowrOvzEwd1I5AJijT+6UOf6oLPnZdWkEpVxZDZqHNOEXwWjUnCN1QPQ0JW3Su23JUWPGsyV/eXeXXuTcKsFrs3m8Z4Ooe4iT3gogFUDFd9ovq/2jLzpSmEH6lmZPhfiQXC8i8CAR/idX64fd84sB0D9iLaXWRCkPkgWUIo3eIBfSlFlV1Mm1WL5L0fDlhskN0Fkl5WhysyirBP0WeZ/sZPa59nJRiirDsS3FFsJ+veNdx0+LyH6+kzuikzHQBMz9IrqmT261vDBC+/fuK50IDLy8CBJOa9SDUKWA7CTSHNqz3xRARgJkKOp2hScPjoQBMYy9NoBSdDwegH+X9ccq4R+LfnPteGbiEFCImwTgCCz1duFcAproyYWkBb7CDMFuyTEwF5Q60VLf3qRTWxWuc2an7L5IJSkMeVQu239PRIWvuILEcijkEW6OA9Bi6MRFn1TNTpQL5QAgZy+arl87zARp5TPHV+JL+8gxJMLfvCdCibdEaDWgPzgpYvkVuXI4HPhZ0gT3V5aihoqibB6KQVGHz4la7U2USBz5XWcIkI11NeSZBisktN1C4LiBBe9kMMa8UTmDumFQquzcBJYUle2LDpLTYwZKhHzvKUMtZnnTnlhiIG26a31KwI0W/wxR38WcA5RuxyyfBix3Yl/Uovt7Wd6ub+58ticJT7E9DThiWxV+FdRSKs546bJXAlwWcvy8ehwvw7gWQCyCKBosm85QDUayoVFFBeLX7L5s8kco8zY/dUeRTgdrf+ByCm5MD4cuoDIR+rH4Vy0CEFm7i5JqZEhSsEEAp66eqdBCjekNkXvV3P9V5IMKQ7KrUDY9eCVdsLddc8/vixJ8l1A+Whrj1UW5BBmfAO7VgCK+ibbywPfNfmBk5j3wAfxdFNgsUycGkeWysKlc9oXgB97tXs4YYV26hIlMpi/joiHezdt36dk04MtQsTEmmcRENvC7Rb0Xnaih/sbNc90vphNkl5hNSPNbe+5LTm7G/kOb5uTp1X+Ih/PNB7Lq1F9U2oNAk8Z+JabZaHShI0lHSGiKGJL0+UkcKovR0Q0dkU/PMUggOXtm3CVSuScJ5pkTrQK/yp3yJRehYxmMBZ4Ms7c+14F5J64Ur9TKKaVuSMbHvS9q3p1CrG/QpN4a0a5CfUOKzttW1LtEQuAYhKGBo8fI8vYISaKPNXYQkFe63wP0jFrFyez2kOG/dvfoZCW8hJbjR17rqu1lFMauBQglSqETer+YpuJJUnoZvnYJJmkPwh/l7+FOhRRGUROrhFFNoV2JrqnBmUBB/iCS24mT3YsSdCQgjZ85YSenUL/bYkkyT8OcJVvDnMcETTp9DGp1qB4JlCnYgJ1g7hW43rfZCpwvnrm3cBjKpn5Ixv4ncOBbzQqZlexzFjXVQ2gbWOoXNebOw2GDzv3MZk4Hl3MLjNC19AtdxGwCLQbsjAhM8VdgG2NQjmB9tin/s8l3VKSJij4SVTuc4tQkDuh9bqDseDXlJSKq4rEyGZ1+7wxLOBO9Lc2JY2pp2vM2RwxbC71vqgTzJqr4x0SdN6drBkdCPJBdQeQEZUO6bnkZr4FMmEM7xwOcK0wvgc6zN0LJ2GAWoYkqzZMhIIk+B54pi7hFJqXjgGQDRU9wS9IU2FllX9tZxdE2HjGkQ6pKgD8Z84jRXuJ1QF4aUZouLBwZAYRXMQKWvEyB6gtCXEDWLAkkq5CMk4uij+igEP7OwSfUYfV2tBLahbeShrxwIcR3F3KthyzqPDqBuPhQbjPHsia1M2F5nggTNC6UZoKyy1xgSUArGLGwr14py5FDn3WlSleq+UHkj20DZ1Wni3j5uqYrZK8qmrrD9K6gkFzSElrFW6Cekw/a6PRT96qaujxtfs/LXlVt6pbyJ8i2yYHR5aS6OozuBerBw/2S1XV1TneU1IZK4NT3a3Mzn/3AK9G3HECxyEPp3bour7ruQw1ewt4T553Aa2dnBtW158dyYM1+X+QFXhI2xy4XswXi73jJoVrYvRf5sevBgXwP0w60uaTGjL5rSo33He/K05+Hpm1h0ism7Q3wTFUp9YFeDeRXXQz/GpLuCbCWcO4aDRZdKa5X8dlrU3krEEMQi5+94QyvWO95/gg5bo906zq7h2rqUPHuEdbGQv9KXRPK63wYHL8LoJ3NvCEpOp7FYodxxXvKIOGxB8m6poHg3LfgMjaxVr4KmtMPDRLzskCThzqNQPoNnik9vY3VKLhqE9ANo3iZEuHtvBn4hhL1SfQw+P+MHkOG/n38h5r4pPcW0SdD73OcmmcYdJmiySqiWwOpLlvpnEhl4ejWauNOa8Obtj0WAFChTt/xE238CAHNrJM666j+LxBz28o2sQo2yIyHG7Vw7jf7xw2h4Sc7rvd8YwS0cxBNIWLkou8lpv0uYYwZ/H/wq/T9Um69loHTiB897xkdAsuMNQY1apcKejGm5+/1yp/v/3tW4BpGqT2ZdWtWyx6WVQxVUawvfhFSMfa7sxGH435fCmod2lFIt9mHpnuEpFWiqi+O0mHBSlRNd9rMOjRn8+TbKLavN95jqdCczkrp2NukjNdjqSi2TouFN3SeaNFN+s8SzzYxqVMURsvkWePHtMckS3oznSFfRNmlxycrCjYYtNAcsqQnOhFO9oOdZCywJGrD5oCUdLYbra9rXnBBpS6GdPMM1DOZCCSjq0R9wjajI+dE0bYpm4eTmlF69QdnZGhaKCJBgs6U7TSSfbvj1V+kIBkkS5AoYV2XQJVedpJ+2en0ju1Ezk/q+DgjqvwBVe6O5WiFsmNmKnsPf+fi+/d/qn9Ec+JHqwfD4cK7JNrEUBY63nQPm9d0m1X21rFt2EvszGqGmR5wVHfoRH8AFyVhBCZZ4KZkloWX2yBKqd8j0troopQK5Gf3aDv4JLCXtCVnBfCZDRgZM25Yu4RxdTDJFfcHriJ4+DYz8VOnHJikwADgmBsnjJ5sMpZmoLOmfIKsD20DDpo0JZMA0wrmjWcykyHrhhit4+B4D0ETw0U687wzCME4tSscY/nktVpiKKOpel5Htwt/BrOVeB2ZzCWYntIYloZKpwE0sa2FsNBa1gA60IyFd3VqcT6PdxVE1vm90kD0BriwrjeG78bfwue9FMhNF9Vu1PsCOyBOQ2FSZ/SsSOvLd+KSdFHvxEcNgCMurk4vYH4iLYtV6gOzoyTo8o2IyX2xvOTUD6HHDzSUZozV4NIYc4C9MKPZnok9cRBnNPhs+InnQku8ji6PQKrDoTMxvWqYBkoO3SyQVnG+u+SCqAKg4RDEDCyektTbfCe2oXLd71MIqikVnEbcBRgF5ctYyO7B+qlhsodivmQDmt7jKQ7OC0JJ5gAZH+gF80TBAbgOARRdF+bdKjzCaDf4Glruxzlyh0J0HEKbR3D1dnRiX+YaXuoWzniw2GaP1rTHOXKML0jACPNjpcHsiGvNVavn4aO7yxCHqHnRQRTXIGXnnknImio4JpOQbkAlaDed0EDuWMB6gY+X8RGPxz0OuDzaPMByOBFh45kYrJGeDdHxXFCeoTCRbcRBFmf87VRy99lx4BRFdZzX0YBmMvz5ienR6PlMfIQTCwmefH1enxIzgxkr/A3h2nmpvLduC5KBYyX8q6iJK4p9/HtsRAAlgZnaCS8okAZdUvwW0p++j5ZLSSzCnh+1RocD3mN8BL+IzzrAf2i2dHfD4caRHJtTlHuk2GJBialZ2CcuDJQnUJnipTnsW7PDaiA+Dvvlb+I0jf5to+hdINh3qDu8N6BWi4qgu0ZIASu65ccfSEC8ErsZObB95j4dUMqvHlH1MsvqZe0a0Vax5tF5BRNsfCbTJJKNhN8dq1YLv4gwMtfDBt+vjSSXOQVdYXSnKStSU44ZqRGAcxTtUjivadg9D9hYInUU9bsXLEqZaBt6MHEt38bgu4cJDnE8a4fYa7kpw0LrNnhmKat47FBgz3OuUW5wF7LXGb5cIzQigmFsSOgjPsjVI+5LCFPazKEZADZNwBZQc+tqgDkCnlpRToVIQwCJ12MG2qtusnvZjCWEDIJ/EuMIqxooi5vO0cwYzawYIstl51ABGuoPgulddiUBdzZeRKvUcxABpUNzxHQRvq6Dlw9IVfkqXqLhniTuUuLCKPgsYv+T1NWv6duv7z7/NnoUAj15lItugMwBgFEqtMXg7l/eYmijdVrupVSJ/dlDnmGLBVxm4olzQW89+AkT+VmWc7CIxFQOnRLFvClwDEJ1/pS6wQ5GL0spc0+m7BnfVcnnqNgSHhpWN7UI/BX4QUoA1Lsi/bRKmqVoacx7S4WjC/uiSkC2LTB7SNzG4ULa9OY6nX5rNfsTJ6f9mf6fv9JSmdS/4lMt6WfgaO30TtFfb8L08GoGgW53PvBuFzy6ACPGDY3+PSSEd1F0M0LT8KUUtbfFE1pST6Br9QtKzPjI/BjuDJvsY/n54Vhma8zIR3IO4BJrH1m8eTk09abLVyadhtcY9twWsX7vsZl58GrbzBOPXq3NzVqvcuSR5tOusB3dRE/GJNKbouI4r56pyOG75GOLN20afiEd04b+P8r0Kpni7bEDa1DSCzK1H5p6KbFcDcp3Gdav019ugtk1H7yOiMzuKVatJVJYXypzpZrX23b5nIW2FH9n4+O9emW5CiiG9ZLcE2uDZNwMHw+0/XZ1t4UyB3YlrJvdAL2eCMxT8EbOmTAettHcMLw+E8Gd6th80smlevQIek+v/NDspLM466S/JldjOon7NOJMp0BrFcGcjCnU1WTutfbcwizuZNq19o7GLO45s/oX3T/dstO3weto1oxlCULtDqpFJmxA+vKeP4kk3FMn//Z8gvIpU7nPDB23XhjnaF49Ml1LKZBniqmXJ6CJy5nFkeVW6jIzCkgOd5PvEz9NjqpNp9yWnTG+v9ewxvju+7PnTHmikea/zJabMJ5Nx0PhNvkQQfIHFcDj1VTOLh8X0UMihrE0oUdE47dF6heLOKha9BLg2+7hiJnojzST7EQPQbql90vxTxg+4Ih0HT0hjTBZXdKly5EClvNPMGTKyuQCGd/tkBuinMTLJfYYl3h9AhUR/bZBvvFS/1zDzklSZghA8FqCuF+Kjupa6mb9lxKhmwZ8e9yAsfSbhK4ermFE/bQ5fSlHmhFwCC9BxduApbkicGmoK66N/jcWZghAxbJ0rw2mSNw8wwJk6kvq406ufxYZU9Sl7O6+fGX1euALME1JPsHvzeosKjaa3Y3HxBr3nf5ZjNTSUQNnTwLlvZoL+jGXwaaXb2exS3MCKPFyRDc/+pzBrIp6Hts+PZ7TOvV3l7IBPkvjq+e3nbriMwSy1TOyL03ve3oTz2+/7Z6A0z1P6nb1ElKoWtlQ/0LNOHdpU1aWl/xAH0CDJ2a+PQG8KJmoD3hn5ZkhYZ73Zapzr3Z2hvPVM1tCNz1TCvz66/PenLqggMlzGWkwiYF8AjzfebbrZmnvJpf+FeYZavq365KoGyxV/KyogxT8YNs+2HUxEAXyKH11S09WGUMCjKlnq5La1d8AamLpLQ==", "experiments/gave2_ensemble/training_utils_v2.py": "eNqVV01zszYQvvtXqJygxcTORw9u3Jke3k5PPfbi8TAyCFt9QVBJJHE9+e/dlcSXATv1IQHt16PVs6slk2VB4jirdS1ZHBNeVKXUhApRaqp5KdRi4dZEXVRnQhUR1WKxSFlGqlJxzd9Y/M748aRVnIG3OClroZW/IPBrNELzpsFnbh8LLnhRFxuS5SXVZEueo5WT0I+h5HGFooAsf4XIkUiplPS8GbiPzRroggJV5sVvQ5NUnyu2BZHx+PNz0IGZMrQoZ6x4dhU0UidaMfLDtu/QLlqM+JOUK0b+onnNvklZSt9rnECmU2tJbN5IUStNThRE+sSIogX8QW9eCwCxim6Dbg+vW7IKSCkbcX9/IBtqB7ewfXtj8kySE5CA5T08A8yCHal5qfgHy5VD54gACR3EX16HJw9XK8ZaMmChwA0kOa985yxsyBI23AjgrPB0/OZ0nh4DR0nAKemRxZVkCVfAX8vJE1e6PEpaqC5t7Vrojq5dMFxLeaJ3SsuQlIe/WaL3Q8pNk63v9AbtJgh337JHvSvSuVc4+1ZBpLxA+eOtg/6jzYo95QNw7r0k7J+a5vnZ0i5tiLA8nJcHLojBrGbZSF4naAiLjTz4EqCmGKhkhIs3mvPURWz8IMe6dNSFD9RQ27VTMpwkbXquxGPUalQ+zkWvctT/KBrBWKpmCoZ90KLKGabQ+Evqos6NKB6SCwSIvFncbUKy2SzX+5D0N9M3lwxZC6wfODBJmLFu6wQspoA8oB9Xdv5kqJCs2XL9GLgKToA6s77aVCKaP0vB9j2rmItEsoIJbdGnPMt8K2owh4i3YiLF4viXyVL5TsOWwG4F21sHgQUz6gXWr8lpu/TjKPooQUA/JhLWI5zCvRiCGJ2cZ4h5HG+Qvc7TVcpM07u03PJGfjx3GWK/KxgV/kgjCMIb9rGjpQJHI2GkyxzKzu976KCOQneiYNqiH6xbnYoyxomJHEXExeC+XT8uvl9F/GwuCKWYUnHOqBRcHGMoSmZnFdOA5Rmtld6N2v9+MLjErCqTk9pAd8KjX78MpSOI3TizitarO8pmP53FOnppBqCZS0kyqmBUc9BBvgez3b5tdqVud9e1MMc7r/wOGfud5gp46TlPsLLzNHQ5zFFjS7girKj02dt/2tZTYiPTLHXZgKBAdh9S4svyPToy7XtG4oXQXAOSQXsFAeSscdn245Gr1+tM94AbiBGtsBP4mVcKuKwu1x4+ifPUCn4xbZlcho4/3cUCwFT8zvUpphVmD4GOASPWiQJFDVTERj/Ha6czOJV+0NkdNg57zbpgWvLE3o+1oG+U5/SQNzPiV47XPcHadeJAOMqlvakOTGkAGuNO7VH38YfkOztvc1ocUooba+oYHncTKdu7Lu2cgkOr3gsybdZZub6LQG6FsunfD/jXB96SsIHyeqOO52mI1mR8VBfndRM9Z59YQgeWA4bLfAhU9IagzE5f73WMOWztOv5mgZoYlzbcJnr6Gt5O22sDNaONbclGA7tD72xtd7jXyrv+MPAEqLB6cIIwJQfYHAPecCSzJ20e8awHtgGkcR09rm5UmyZwO0CK0HtZ66rW7Vx3gs/fmczZNKFrb+5iN6Voyt4VXycaF2Yn+0KB9rRtjidGCJf8u6rNLdwyYXiBglotFQ52SqOd+0rzzcRu7kNzU5mbyJyJu6NMkwcVzP+tQfq3fvrbYKQ328LpS/hG4bDSI4jxvd2Sp9EVt1tFj3ABRU/458WNnPTdfYRJKo7Mh9HSeviJrOc+wNDmYYv/zAw5OOXdPP1Af7/4D2jt00s=", "tests/gave2_ensemble/test_cmrrwnet_v2.py": "eNrNVltr5DYUfvevEH6SwKsmzi4sgYGW7JYWNskyJO3DMAjFPs6I2JJXkmcm/75Hvs4ts+lLqQmxdOZ8536xqmpjPVHtq1RPvPGqjLorabTyHpyPCmsqUku/Qo6el3zHaxRF8/v7BzJrb1SIQpUgBOMWnCnXQBmvpQXt3SJdRrf3Xx6/fRXff3v4AxEt8BcSw7YGq6rAFIf7s1xDKkA7qJ5KaElZZe1GgxfrlNevMWrNSukcubmdz/++A/9X+oBWOjrYy8P1Rjpg1xHBJ4eCBDriRWXyBm2ErUKEkDoXuQEntPFC5muwXjkQhSzLJ5m9UAdl0QsJT7hy1IxsD7YBuuMR7yRSxiZu09gM0NVdNgsyFx62noLOTK708yxufPHhc8xOqflT0/jGWAuZh3wOWWOdWsPgeJz0Sk5i70wLH1FzKJSGEOn3QjE8ZC3LBtDsH43CrIpnK/MdbAv+dQy8e1H1oy7BObpfUhxV58LVkNHYG5utYkaUIxh2cmc0JKSjDjSlncccQN5HZcxgtgpk/QzC2BwspnONRintwWpZCrygF4FLa3x7g/YC6MM89iXsXt0hqbUimqgFcd7SUKusNyygeOiFSVwbvJ7K0XQMIL1IJuQU4raRdiqe71c736n0waJDj4OI0eEdS3sKllvrA/co1Fi6wOeSXyyXCVks0uFwFQ7L5WTYqAXxZzXS4cCiU4Xz9UeDPKOExXVCLpZceagoSwgawt6JupxQV+9HpRMqDaj/ujz9xgg7tpkTtcVxk4HwKyzDo+KtLeQq88po9y8qFKo6jNlj+tZvrKz//xX99jybzNwovxpdxXkekNK+flEBaewrZUQ64qv6wOZh5Lb7CH9mYX/gzIfStavjmJlvLJZMN5L3fg7PEFSeQ47eHTOEJ47jk/S9BPyMgWsdPNI6OsnaLbxHjNFtu8Co1rw7seuTgKEsBXau8kK0BZZgtusmTNGEmMYPxydclXg4I6kNV4NJxo0+SmTn2UOjjkowJeP5rL2FsRtp897c7U9ssuAbq8k29P41jgwLNUhPL5MD9ThP8I+di+0dbHbCOx3PWFAj7t0VwZIj0uEXwD7HfnjbGsYovt08J6pXupdZHP6n8bH2PuvtpnSzj8cMuql2h9ksPWaROL3WMkwwlATZS21wPQSPfpelg2N+UxQqU7iou96bda9zfncpdOh5GwHaNQvOZBey/CkhnzGxKTu3jXDk0l5M2Cbszc85nM7UNzWmv2PnbiVrYGQ2I0HZ1aAsFGlvWBifg2y0IVKh47Ss8Cs4wGIhKqm0EHFXReMmClRsoH8Am0xl9w==", "tests/gave2_ensemble/test_training_v2.py": "eNq9WG2P4jYQ/s6viPIpkSAXwrJvElKrqu2XqldV235ByPKGCfjWsVPbsOWq++8dO+8QYFe9K7pbEnvmmfHM4/EYlhdSGY+5L86eo51hfFS+evqg60cDeZExDvX7TjBjQJtRpmTuFdRsUbmC8X7D11EtKXZ5cfCo9kQxGo1+//jxyVs4iYAQi0hIGCnQku8hCKOCKhBGL5PViGWeNiqwGqEnJDoprEeRNfY48vBTv0VMaFAmiMetRojGUk619p4UZYKJzZ/JEzqsg9r1yL7+QDWEJdoaMs+OEyEFkEJBoWQKWqMq+QxKgiawB3Ug6ZYKAZzIndFsDURJFmjgWYVjPy4q8HcBiuV2PdGG7iEhgH7mzxyiHnodtnaQ5HJNOTMM9KjBTLMCIyeKKNtxHgTTZOxNb8feLBx7cTQfe2tzKGBh57mkZpaErTcZJXRId+p0Z1d19+eV7y4pY2RKTRs/3Ve9qLVMHicoO3uczFaIMI3iNg4Y9mJncHQwXgGGaWwhxqXr1dd+7KEMLHybXD9s0WziIqQJ8ufHv3aUByV8pLe0gLHXuDwPw/M6bgmV5hJZGK8ivcuD0EUoHlT8WQE1oPqqmMX5kWqfmoXUuMo9kFdgm63RxKbkQJ4ppyJFJuLuIZlUsFFyJ9aE4v+Us+K99DTVliG2GmiyTxqKnthHJJKiMdNhajVpU3RBPmjk7aeWtIygStFDsLyJMJRJbP8iAVbhuKdgpKG8Iz2NS8nO17FKjovKd/nCAvcn6N9uwhlrZjosQTM2/hiSKn2Ecp5yqSGoVjb2WldKnxv30Q1PGckXU5jcHmd0y7SRG0Vz5CgounGVJ2WaSUHWYCDFoGkscoxyTK94QRe+WjZPLbr0NC51Ulqnp7+dcZEnO/n2puW7y9H7VGpDdhuVez/uw5X7CyceBiZqjWYmB6NYaql4dbFBbXpc4g0Xie95LnW17SvwpX+KXZ0R2l8t45Xjb/hmNETZUw52Qx/BxNE0/A9OcZaZE88GqgwHqhxhNliiiIJPjoXyGU3tASuKFNpQgdVIMamI3iKX0p35erTENWFN73nRKf+VG5jSf3qb2IdCplv/EZfU392+lrjuNUsBJ3G59zfz+9uH2ez+ZhbPk2R2JG0FiYnnKDyLpg+zm9uHeTK9m93ezeYwiedH4s82Yo3pY8vIIyhnNU4/tNNf2uMO3KoXg+sOlvV6MVdV/arwFtN4mKI/UY51qYRd+vLFXw2S5kntIKDiENRM8Rqm+LbXwgNKS+HhcVI/usEStRxBEoWXyUPTFAokT4rH87PCMoY0ZGINBeAfpNA3K20XOeSGL1LoOMsnm8lRKUmuibk955g0f5Nouz0fvWUS2XJp+7Mkul9dJvXDeRbHx8edj4dXt01oULo758voPGkGiVoFdYCn87CmYZ8qZqvA1pd0p7RtEjAU6IrliCZbjA9O7Rm8YsnBfbRpmoivRpLWcg//Tcf+Gd3AtuNLZEbVWGNnvmpP/7ur1dv2fueg57YpnJ6Wa7c+2+dhw5DRHcftZvtAfkM0zeCd0apjU4Luk07hVWzDhDvP7cWLqs1++bhqz2B1eOyxrBZC+WVvwlGwxo+Kgz8+nZ5M1tTQiZLSDE3byWE1tROTNVNDkzilP5SrTXOlXgUYdGAYJpN8PcmpYBlG+O1gH6weqfWiT1gjh/EN1S9DM3b8gktDM/HR4Kr3hhmw3U8bbqo0lmUcDTqXPJtXfj5/deovXJssYvSMt+mmgtl7U3hFQWCVUIDmwXERC1141YRJt0Szz2CvkVeEsa9b2yNol78BmeZ4cfSfs+mtf020LGzoQBJfE+UKTcPk5rocsUeh7fos7lUPqMJ7H7auRWErW6s6u+pRe23G5Xavw981v4zoF1b8ITiKBP1fhiJMFVbjAtLAN1LhWRl6TLtfZ35FJAR0o/UYs40i57CuItqUrOrCuiZYSzXBZgDbg8+AVZ/hoYgAB7Jnkrt24aTc1wXKGhq9+RCwhkDXyntcGvCOOTvdgmH7zWx34GzgESC0VMESPxO81K2woC+XzUM5tOr0WEyc1T/SaoC66jmWgEYXg6rtpan6N9zw/WITNbikwHkydph4eAzLVP5WUv8/GTKJ3SEx2A/plwOWg1eq1s2lF1meSoVHoum2B9+CE303HDFfmdni04Z1mwKDuwjMUHan9sZvL/yuBaieL+cWm74XCErIVqxa8ZCNucWdzEsjk/atZ6a+oTUA7vZd2qqwLzCpdwpcjkqNNq6iUjPtfRCls2cx7E+5LPMIETQHQrzFwvMJHrJ4oBG/5EBDVjuKB9u/fhwN5A=="}
patched_files = []
for relative, payload in EMBEDDED_SOURCE_PATCHES.items():
    destination = WORK_ROOT / relative
    corrected = zlib.decompress(base64.b64decode(payload))
    if not destination.is_file() or destination.read_bytes() != corrected:
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(corrected)
        patched_files.append(relative)
print({"corrected_source_files": patched_files, "count": len(patched_files)})

In [ ]:
requirements = WORK_ROOT / "experiments" / "gave2_ensemble" / "requirements-gave2-main.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime"
assert torch.cuda.is_bf16_supported(), "The selected GPU must support BF16"
gpu = torch.cuda.get_device_properties(0)
print({"torch": torch.__version__, "gpu": gpu.name, "vram_gib": round(gpu.total_memory / 1024**3, 2)})

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
OOF_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
run_module(
    "experiments.gave2_ensemble.integrity_v2",
    "--data-root", DATA_ROOT,
    "--run-dir", RUN_DIR,
    "--seed", SEED,
    "--folds", 5,
)
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests/gave2_ensemble", "-p", "test_*.py"],
    cwd=WORK_ROOT,
    check=True,
)
assert FOLD_MANIFEST.is_file()
print("Dataset, folds, and packaged tests passed.")

In [ ]:
MEMORY_TEST_PATH = WORK_ROOT / "experiments" / "gave2_ensemble" / "memory_test_v2.py"
ORIGINAL_MEMORY_TEST = MEMORY_TEST_PATH.read_text(encoding="utf-8")

def benchmark_profile(task, base_channels, batch_size, activation_checkpointing, steps=5):
    source = ORIGINAL_MEMORY_TEST
    replacements = {
        "train_ids[: max(2, args.steps)]": f"train_ids[: max(2, args.steps * {batch_size})]",
        "batch_size=1, shuffle=False": f"batch_size={batch_size}, shuffle=False",
        "activation_checkpointing=True,": f"activation_checkpointing={activation_checkpointing},",
    }
    for old, new in replacements.items():
        if source.count(old) != 1:
            raise RuntimeError(f"Memory-test compatibility patch failed for: {old}")
        source = source.replace(old, new)
    MEMORY_TEST_PATH.write_text(source, encoding="utf-8")
    for bytecode in (MEMORY_TEST_PATH.parent / "__pycache__").glob("memory_test_v2*.pyc"):
        bytecode.unlink()
    started = time.perf_counter()
    try:
        run_module(
            "experiments.gave2_ensemble.memory_test_v2",
            "--data-root", DATA_ROOT,
            "--fold-manifest", FOLD_MANIFEST,
            "--task", task,
            "--base-channels", base_channels,
            "--num-refinements", NUM_REFINEMENTS,
            "--steps", steps,
        )
    except subprocess.CalledProcessError:
        return None
    finally:
        MEMORY_TEST_PATH.write_text(ORIGINAL_MEMORY_TEST, encoding="utf-8")
        for bytecode in (MEMORY_TEST_PATH.parent / "__pycache__").glob("memory_test_v2*.pyc"):
            bytecode.unlink()
    elapsed = time.perf_counter() - started
    benchmark_images_per_second = steps * batch_size / elapsed
    return {
        "base_channels": base_channels,
        "batch_size": batch_size,
        "grad_accum": 1 if batch_size == 2 else 2,
        "activation_checkpointing": activation_checkpointing,
        "benchmark_seconds": elapsed,
        "benchmark_images_per_second": benchmark_images_per_second,
    }

def select_cost_profile(task):
    desired = BASE_CHANNELS[task]
    fallback = FALLBACK_CHANNELS[task]
    specifications = [
        (desired, 2, False),
        (desired, 2, True),
        (desired, 1, False),
        (desired, 1, True),
        (fallback, 2, False),
        (fallback, 2, True),
    ]
    candidates = []
    for base_channels, batch_size, checkpointing in specifications:
        print(f"Benchmarking {task}: base={base_channels}, batch={batch_size}, checkpointing={checkpointing}")
        profile = benchmark_profile(task, base_channels, batch_size, checkpointing)
        if profile is not None:
            candidates.append(profile)
    if not candidates:
        raise RuntimeError(f"No safe L4 profile found for {task}")
    return max(candidates, key=lambda item: item["benchmark_images_per_second"])

TRAINING_PROFILES = {
    "task2": select_cost_profile("task2"),
    "task1": select_cost_profile("task1"),
}
BASE_CHANNELS = {task: profile["base_channels"] for task, profile in TRAINING_PROFILES.items()}
(RUN_DIR / "selected_memory_profile.json").write_text(json.dumps(TRAINING_PROFILES, indent=2))
print("Selected cost-efficient L4 profiles:", json.dumps(TRAINING_PROFILES, indent=2))

In [ ]:
def train_fold(task, fold, epochs):
    fold_dir = RUN_DIR / "cmrrwnet_v2" / task / f"fold_{fold}"
    profile = TRAINING_PROFILES[task]
    arguments = [
        "--data-root", DATA_ROOT,
        "--run-dir", RUN_DIR,
        "--fold-manifest", FOLD_MANIFEST,
        "--task", task,
        "--fold", fold,
        "--base-channels", profile["base_channels"],
        "--num-refinements", NUM_REFINEMENTS,
        "--batch-size", profile["batch_size"],
        "--grad-accum", profile["grad_accum"],
        "--workers", 2,
        "--epochs", epochs,
        "--amp", "bf16",
        "--seed", SEED,
    ]
    if not profile["activation_checkpointing"]:
        arguments.append("--no-activation-checkpointing")
    if (fold_dir / "last.pt").is_file():
        arguments.append("--resume")
    run_module("experiments.gave2_ensemble.train_v2", *arguments)

train_fold("task2", 0, 15)
history_path = RUN_DIR / "cmrrwnet_v2" / "task2" / "fold_0" / "history.json"
history = json.loads(history_path.read_text())
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from experiments.gave2_ensemble.training_utils_v2 import assess_learning_gate

gate_report = assess_learning_gate(history, minimum_epochs=15)
print("Learning gate:", json.dumps(gate_report, indent=2))
assert gate_report["ok"], "Task 2 fold 0 failed spatial learning: " + "; ".join(gate_report["reasons"])
print("Learning gate passed:", history[-1])

In [ ]:
for fold in range(5):
    train_fold("task2", fold, MAX_EPOCHS)
print("All Task 2 folds finished or resumed to completion.")

In [ ]:
for fold in range(5):
    train_fold("task1", fold, MAX_EPOCHS)
print("All Task 1 folds finished or resumed to completion.")

In [ ]:
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
from experiments.gave2_ensemble.integrity_v2 import certified_checkpoints

certification = {}
for task in ("task1", "task2"):
    checkpoints = certified_checkpoints(RUN_DIR, task, expected_folds=5)
    best_scores = []
    for checkpoint in checkpoints:
        history = json.loads(checkpoint.with_name("history.json").read_text())
        best_scores.append(max(row["soft_dice"] for row in history))
    certification[task] = {"checkpoints": [str(path) for path in checkpoints], "best_soft_dice": best_scores}
(RUN_DIR / "fold_certification.json").write_text(json.dumps(certification, indent=2))
print(json.dumps(certification, indent=2))

In [ ]:
selection = {}
for task in ("task2", "task1"):
    candidates = []
    for tta in ("none", "flips"):
        raw_root = OOF_ROOT / f"raw_{tta}"
        run_module(
            "experiments.gave2_ensemble.predict_v2",
            "--data-root", DATA_ROOT,
            "--run-dir", RUN_DIR,
            "--fold-manifest", FOLD_MANIFEST,
            "--output-root", raw_root,
            "--team-id", TEAM_ID,
            "--task", task,
            "--mode", "oof",
            "--tta", tta,
        )
        task_name = "Task1" if task == "task1" else "Task2"
        probability_dir = raw_root / "cmrrwnet_v2" / TEAM_ID / task_name
        calibration_path = RUN_DIR / "calibration" / f"{task}_{tta}.json"
        calibrated_dir = OOF_ROOT / "calibrated" / f"{task}_{tta}"
        run_module(
            "experiments.gave2_ensemble.probability_calibration_v2",
            "--data-root", DATA_ROOT,
            "--probability-dir", probability_dir,
            "--task", task,
            "--output", calibration_path,
            "--calibrated-dir", calibrated_dir,
        )
        report = json.loads(calibration_path.read_text())
        candidates.append((report["calibrated_dice_mean"], tta, calibration_path, calibrated_dir, report))
    score, tta, calibration_path, calibrated_dir, report = max(candidates, key=lambda value: value[0])
    assert score > 0.25 and min(report["calibrated_dice_channels"]) > 0.15, report
    selection[task] = {
        "tta": tta,
        "calibration": str(calibration_path),
        "calibrated_oof_dir": str(calibrated_dir),
        "report": report,
    }
(RUN_DIR / "oof_selection.json").write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))

In [ ]:
selection = json.loads((RUN_DIR / "oof_selection.json").read_text())
calibrator_path = RUN_DIR / "task3_calibrator.json"
run_module(
    "experiments.gave2_ensemble.biomarkers_v2", "fit",
    "--data-root", DATA_ROOT,
    "--oof-task2-dir", selection["task2"]["calibrated_oof_dir"],
    "--output", calibrator_path,
)
task3_calibrator = json.loads(calibrator_path.read_text())
print({key: value["accepted"] for key, value in task3_calibrator["targets"].items()})

In [ ]:
for task in ("task2", "task1"):
    chosen = selection[task]
    run_module(
        "experiments.gave2_ensemble.predict_v2",
        "--data-root", DATA_ROOT,
        "--run-dir", RUN_DIR,
        "--fold-manifest", FOLD_MANIFEST,
        "--output-root", OUTPUT_ROOT,
        "--team-id", TEAM_ID,
        "--task", task,
        "--mode", "validation",
        "--tta", chosen["tta"],
        "--calibration", chosen["calibration"],
        "--accumulator-dir", WORK_ROOT / ".prediction_accumulator",
    )
TEAM_ROOT = OUTPUT_ROOT / "cmrrwnet_v2" / TEAM_ID
print("Task 1 and Task 2 predictions:", TEAM_ROOT)

In [ ]:
run_module(
    "experiments.gave2_ensemble.biomarkers_v2", "predict",
    "--data-root", DATA_ROOT,
    "--task2-dir", TEAM_ROOT / "Task2",
    "--output-dir", TEAM_ROOT / "Task3",
    "--calibrator", calibrator_path,
    "--split", "validation",
)
assert len(list((TEAM_ROOT / "Task3").glob("*.txt"))) == 50
print("Task 3 predictions complete.")

In [ ]:
report_path = RUN_DIR / "final_submission_report.json"
run_module(
    "experiments.gave2_ensemble.submission_v2",
    "--data-root", DATA_ROOT,
    "--team-root", TEAM_ROOT,
    "--output-zip", FINAL_ZIP,
    "--report", report_path,
)
report = json.loads(report_path.read_text())
assert report["ok"] and report["counts"] == {"Task1": 50, "Task2": 50, "Task3": 50}
print(json.dumps(report, indent=2, ensure_ascii=False))

In [ ]:
with zipfile.ZipFile(FINAL_ZIP) as archive:
    assert archive.testzip() is None
    names = archive.namelist()
    assert len(names) == 150
    assert all(name.startswith(f"{TEAM_ID}/") for name in names)
    assert len({name.split("/")[1] for name in names}) == 3
final_sha256 = sha256_file(FINAL_ZIP)
print({"ready_to_submit": str(FINAL_ZIP), "sha256": final_sha256, "payload_files": len(names)})

## Result

Submit the `FINAL_ZIP` printed above only after the final cell reports 150 payload files. Keep `runs/gave2_cmrrwnet_v2`, the OOF selection report, and the submission report in Drive for reproducibility.